# 09장 보안 실습 — 로그인 아티팩트 교차 검증


## Goal

합성 로그인 요약을 교차 확인하고 잘못된 입력을 검출합니다.

[교안과 분석 질문](../../09-testing-debugging/09-3-login-artifacts.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-09-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'login-review.psv': 'artifact|user|source|time_kst|meaning\nbtmp-summary|analyst|192.0.2.10|2026-09-10T09:01:10+09:00|failed\nwtmp-summary|analyst|192.0.2.10|2026-09-10T09:02:00+09:00|session-start\nlastlog-summary|analyst|192.0.2.10|2026-09-10T09:02:00+09:00|latest-login\nutmp-summary|analyst|192.0.2.10|2026-09-10T09:10:00+09:00|present-at-collection\n', 'auth.log': '2026-09-10T09:01:00+09:00 lab-web-01 sshd[101]: Failed password for invalid user guest from 192.0.2.10 port 50100 ssh2\n2026-09-10T09:01:10+09:00 lab-web-01 sshd[102]: Failed password for analyst from 192.0.2.10 port 50101 ssh2\n2026-09-10T09:01:20+09:00 lab-web-01 sshd[103]: Failed password for analyst from 198.51.100.8 port 50102 ssh2\n2026-09-10T09:02:00+09:00 lab-web-01 sshd[104]: Accepted publickey for analyst from 192.0.2.10 port 50103 ssh2\n2026-09-10T09:03:00+09:00 lab-web-01 sudo: analyst : TTY=pts/0 ; PWD=/home/analyst ; USER=root ; COMMAND=/usr/bin/id\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 아티팩트 4행, 성공 1행, malformed 행 검출

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 사건 수와 아티팩트 행 수 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $2=="analyst" {print $1 "|" $4 "|" $5}' "$COURSE_DATA/login-review.psv" > "$COURSE_OUT/login-context.psv"
test "$(wc -l < "$COURSE_OUT/login-context.psv")" -eq 4
printf 'artifact_rows=4 not four independent successful logins\n'


### 2. 인증 방식과 세션 기록 교차 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F 'Accepted publickey for analyst' "$COURSE_DATA/auth.log" > "$COURSE_OUT/auth-success.txt"
test "$(wc -l < "$COURSE_OUT/auth-success.txt")" -eq 1
awk -F '|' 'NR>1 && $1=="wtmp-summary" {print $4}' "$COURSE_DATA/login-review.psv" > "$COURSE_OUT/session-time.txt"
grep -Fx '2026-09-10T09:02:00+09:00' "$COURSE_OUT/session-time.txt"


### 3. 변형된 입력을 조용히 무시하지 않기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
printf 'artifact|user|source|time_kst|meaning\nbroken|record\n' > "$COURSE_OUT/malformed.psv"
status=0
awk -F '|' 'NR>1 && NF!=5 {bad=1} END {exit bad}' "$COURSE_OUT/malformed.psv" || status=$?
test "$status" -eq 1
printf 'malformed_detected=yes coverage=selected_records\n'


## Red Team ↔ Blue Team 사례 분석

### 사례: 유효 계정 사용과 사용자 본인 여부

**Red Team 질문:** 유효 계정의 권한과 접근 경로가 적절히 제한되고 검토되는가? 계정 이름으로 로그인했다는 사실은 계정 소유자 본인이 행동했다는 신원 증명과 다릅니다. 계정 획득·인증 우회 실습 없이 제공 자료에서 이 경계를 검토합니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 유효 계정 사용의 접근·권한 범위 검토. 인증 성공과 업무 승인은 별개 |
| Command / Observation | last·lastb·who·lastlog의 의미 비교, 합성 요약 네 행과 auth.log 한 성공 비교 |
| System Change / Artifact | 세션·최근 로그인 정보 갱신 가능성. 한 세션이 여러 자료에 나타날 수 있음 |
| Log prerequisite | PAM·세션 기록 설정·회전·바이너리 형식·수집 시각. 모든 접근 방식이 같은 파일에 남지 않음 |
| Blue Team Investigation | 계정·터미널·출발지·시작/종료를 맞추고 장치·키 승인·후속 실행 확인 |
| Detection | 기준선 밖의 접근 문맥과 후속 권한 사용을 검토. lastlog 행 수를 로그인 횟수로 세지 않음 |
| Mitigation | 불필요 계정 정리 계획·접근 검토·세션/인증 기록 보존·계정 수명 관리 |

**반례와 해설:** 관리자가 점검용 장치에서 접속했을 수 있습니다. 같은 09:02 시각의 wtmp·lastlog 요약은 별도 로그인 두 번이 아니라 같은 로그인에 관한 자료일 수 있습니다. 기록 차이를 모두 삭제 흔적으로 설명하면 회전·미기록·수집 시차라는 원인을 놓칩니다.

**제출 과제:** 네 행을 시간순으로 정리하되 사건 건수로 합산하지 않습니다. 실제 바이너리 미검증 범위와 추가 필요한 세션 근거를 표시합니다. 관련 [Valid Accounts — T1078](https://attack.mitre.org/techniques/T1078/)는 계정 악용의 행위 분류이지 모든 로그인 성공의 판정명이 아닙니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
